# Mistralethon — run the whole website on Colab

One notebook = the entire app: **React frontend + FastAPI backend + Sentinel-Loop pipeline + fine-tuned Timestral**, exposed to the internet through a cloudflared tunnel.

Run the cells top to bottom with a **GPU runtime** (Runtime → Change runtime type → T4/A100) if you want to use `finetune-timestral`; CPU-only works for everything else.

- Colab mounts your Drive at **`/content/drive/MyDrive`** (no space). The project lives at **`/content/drive/MyDrive/Colab Notebooks/Mistraton`** — cell 1 finds it automatically and prints a listing if not.
- Data, artifacts and secrets persist there under `data/` and `.env`.
- The public URL changes every session (free Colab disconnects after ~90 min idle / 12 h max) — keep this tab open.
- Optional: set `SENTINEL_API_KEY` in the Drive `.env` to require an `X-API-Key` header on mutating endpoints.

In [3]:
1+1


2

In [4]:
# 1) Mount Drive, load secrets, point the pipeline at Drive-persistent dirs
from google.colab import drive
import os, sys, shutil, pathlib

drive.mount('/content/drive')  # authorize in the popup; "already mounted" is fine

# Colab exposes the Drive root as /content/drive/MyDrive (NO space — "My Drive" is only the display name)
DRIVE_ROOT = next((p for p in [pathlib.Path('/content/drive/MyDrive'), pathlib.Path('/content/drive/My Drive')] if p.is_dir()), None)
assert DRIVE_ROOT is not None, 'Drive not mounted. Re-run this cell and click the authorization link.'
print('drive root:', DRIVE_ROOT)

# project folder: "Colab Notebooks/Mistraton" (no quotes inside the string; it's MistratoN, not Mistralethon)
PROJECT = DRIVE_ROOT / 'Colab Notebooks' / 'Mistraton'
if not PROJECT.is_dir():
    cn = DRIVE_ROOT / 'Colab Notebooks'
    listing = sorted(p.name for p in cn.iterdir()) if cn.is_dir() else 'NO SUCH FOLDER'
    raise FileNotFoundError(f'{PROJECT} not found. "Colab Notebooks" contains: {listing}')
DRIVE = str(PROJECT)
print('project:', DRIVE)

ENV_FILE = pathlib.Path(DRIVE) / '.env'
assert ENV_FILE.is_file(), f'{ENV_FILE} missing — copy .env.example to .env there and fill in secrets'
for line in ENV_FILE.read_text().splitlines():
    line = line.strip()
    if line and not line.startswith('#') and '=' in line:
        k, _, v = line.partition('=')
        os.environ[k.strip()] = v.strip().strip('"').strip("'")

# persistent data + artifacts on Drive (survive runtime restarts)
os.environ['SENTINEL_DATA_DIR'] = f'{DRIVE}/data'
os.environ['SENTINEL_ARTIFACTS_DIR'] = f'{DRIVE}/data/artifacts'
pathlib.Path(os.environ['SENTINEL_DATA_DIR']).mkdir(parents=True, exist_ok=True)

# NO full repo copy — the Drive FS is slow per file. Import the backend straight from Drive:
sys.dont_write_bytecode = True
os.environ['PYTHONDONTWRITEBYTECODE'] = '1'   # don't litter Drive with __pycache__
sys.path.insert(0, f'{DRIVE}/backend')

# Only the frontend build needs local disk (npm install/build). Copy just its skeleton (~36 small files):
FE = '/content/frontend'
if not os.path.exists(f'{FE}/package.json'):
    os.makedirs(FE, exist_ok=True)
    for item in ['package.json', 'package-lock.json', 'index.html', 'vite.config.js', 'vite.config.ts', 'tsconfig.json']:
        shutil.copy2(f'{DRIVE}/frontend/{item}', f'{FE}/{item}')
    shutil.copytree(f'{DRIVE}/frontend/src', f'{FE}/src')
    shutil.copytree(f'{DRIVE}/frontend/public', f'{FE}/public')
os.environ['SENTINEL_DIST_DIR'] = f'{FE}/dist'

print('backend imported from Drive:', f'{DRIVE}/backend')
print('frontend skeleton copied to:', FE)
print('data:', os.environ['SENTINEL_DATA_DIR'])
print('mistral key set:', bool(os.environ.get('MISTRAL_API_KEY')))


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
drive root: /content/drive/MyDrive
project: /content/drive/MyDrive/Colab Notebooks/Mistraton
backend imported from Drive: /content/drive/MyDrive/Colab Notebooks/Mistraton/backend
frontend skeleton copied to: /content/frontend
data: /content/drive/MyDrive/Colab Notebooks/Mistraton/data
mistral key set: True


In [6]:
1+1+1


3

In [7]:
# 2) Backend deps — requirement files read straight from Drive (absolute paths, no cwd assumptions)
import subprocess, sys, os
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', f'{DRIVE}/backend/requirements.txt'], check=True)
rc = f'{DRIVE}/backend/requirements-colab.txt'
if os.path.exists(rc):
    # GPU fine-tuning deps — only needed for the 'finetune-timestral' pipeline stage
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', rc], check=True)
else:
    print('note:', rc, 'not found — finetune-timestral will need it')
print('deps installed')
# a numba/numpy conflict warning from Colab's preinstalled packages is harmless here


deps installed


In [8]:
# 3) Build the React frontend -> /content/frontend/dist (served by the same FastAPI app)
import shutil, subprocess, os
if shutil.which('node') is None:
    subprocess.run('apt-get update -qq && apt-get install -y -qq nodejs npm', shell=True, check=True)
print('node', subprocess.run(['node', '-v'], capture_output=True, text=True).stdout.strip())
os.chdir('/content/frontend')
subprocess.run(['npm', 'install', '--no-audit', '--no-fund'], check=True)
subprocess.run(['npm', 'run', 'build'], check=True)
print('dist ready:', os.environ['SENTINEL_DIST_DIR'])


node v22.23.4
dist ready: /content/frontend/dist


In [9]:
# 4) Start the website backend on 0.0.0.0:8000 (background thread; cell returns immediately)
import threading, time, uvicorn   # sys.path already points at the Drive backend from cell 1
import colab_app

app = colab_app.build_app()
threading.Thread(target=uvicorn.run, args=(app,),
                 kwargs={'host': '0.0.0.0', 'port': 8000, 'log_level': 'warning'}, daemon=True).start()
time.sleep(2)
print('server up on http://localhost:8000')


server up on http://localhost:8000


In [10]:
# 5) Expose it: cloudflared quick tunnel (no account needed) -> public https URL
import subprocess, threading, re, urllib.request

if not pathlib.Path('/content/cloudflared').exists():
    subprocess.run(['wget', '-q', 'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64',
                    '-O', '/content/cloudflared'], check=True)
    subprocess.run(['chmod', '+x', '/content/cloudflared'], check=True)

PUBLIC_URL = None
def _read(proc):
    global PUBLIC_URL
    for line in iter(proc.stdout.readline, ''):
        m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', line)
        if m:
            PUBLIC_URL = m.group(0)
            print('PUBLIC URL:', PUBLIC_URL)
            break

proc = subprocess.Popen(['/content/cloudflared', 'tunnel', '--url', 'http://localhost:8000'],
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
threading.Thread(target=_read, args=(proc,), daemon=True).start()
for _ in range(30):
    if PUBLIC_URL: break
    time.sleep(1)
assert PUBLIC_URL, 'tunnel did not come up; check cell output'
print('open this in a browser:', PUBLIC_URL)


PUBLIC URL: https://sugar-village-lyric-change.trycloudflare.com
open this in a browser: https://sugar-village-lyric-change.trycloudflare.com


In [ ]:
# 6) Sanity check through the tunnel
print(urllib.request.urlopen(PUBLIC_URL + '/api/health').read().decode())
print(urllib.request.urlopen(PUBLIC_URL + '/sentinel/health').read().decode())


## Notes

- **Keep this tab open** — free Colab disconnects after ~90 min idle and at 12 h max; the tunnel URL then changes (re-run the cloudflared cell).
- Paths on Colab: Drive root = `/content/drive/MyDrive`, project = `/content/drive/MyDrive/Colab Notebooks/Mistraton` (no space in `MyDrive`; in shell use `Colab\ Notebooks`, in Python quote the whole path). The backend is imported **directly from Drive** — no repo copy; only the frontend skeleton is copied to `/content/frontend` for `npm run build`.
- Uploads & artifacts persist in the project's `data/` on Drive (`raw/`, `records.jsonl`, `labels.db`, `artifacts/`, `timestral-hf/`).
- Website flow: **Upload** an incident bundle → **Pipeline**: ingest → extract-metrics (Mistral Large) → label-windows (Shieldstral) → build-dataset → train-detection → evaluate → finetune-timestral (GPU) → **Watch** live streams → **Timestral** chat with the fine-tuned checkpoint.
- The same code runs on the VM (`backend/main.py` + Caddy) — `colab_app.py` only adds upload/pipeline/chat/static-UI routes on top of the shared `/sentinel` router.